<a href="https://colab.research.google.com/github/yoonjae-kwon/minGPT/blob/main/minGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 미니 GPT — 문자 단위 Transformer 언어 모델

Tiny Shakespeare 데이터셋으로 학습하는 문자 단위 GPT를 PyTorch로 밑바닥부터 구현합니다.

**핵심 아이디어**: 지금까지 나온 문자들을 보고 다음 문자 하나를 맞힌다. 그것을 반복한다.

**진행 순서**
1. 데이터 준비 — 텍스트를 숫자로
2. Bigram 모델 — attention 없는 최소 모델 (기준선)
3. attention에 도달하기 — 버전 1 → 4
4. 최종 Transformer — 조립 및 학습

| 단계 | val loss |
|---|---|
| 학습 전 (랜덤) | 5.03 |
| Bigram | 2.38 |
| + self-attention 1 head | 2.29 |
| Transformer (6층 6 head) | 1.47 |

> **런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 설정하세요.

---
# 1. 데이터 준비

## 1.1 데이터 다운로드

In [ ]:
!wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print("전체 길이:", len(text))
print("---- 앞 200자 ----")
print(text[:200])

## 1.2 어휘 사전 만들기

텍스트에 등장하는 **고유 문자**를 모아 정렬한다.

- `set()` : 중복 제거
- `sorted()` : 정렬 (항상 리스트를 반환)

정렬이 필요한 이유: 순서가 고정되어야 `'a'`가 매번 같은 번호를 갖는다.
순서가 바뀌면 학습한 모델이 무의미해진다.

In [ ]:
chars = sorted(set(text))      # 고유 문자를 정렬한 리스트
vocab_size = len(chars)

print(''.join(chars))
print("vocab_size:", vocab_size)

# 맨 앞 두 개는 눈에 보이지 않는 문자
print(repr(chars[0]), repr(chars[1]))   # 줄바꿈, 공백

## 1.3 인코딩 / 디코딩

**인코딩과 임베딩은 다르다.**

| | 인코딩 | 임베딩 |
|---|---|---|
| 하는 일 | 문자 → 정수 하나 | 정수 → 벡터 |
| 예시 | `'a'` → `39` | `39` → `[0.2, -1.3, ...]` |
| 학습됨? | ❌ 고정된 사전 | ✅ 학습으로 값이 변함 |

흐름: `문자` → (인코딩) → `정수` → (임베딩) → `벡터` → 모델

여기서 하는 것은 **인코딩**. 39라는 번호 자체에는 아무 의미도 없다. 그저 사전의 몇 번째일 뿐.

In [ ]:
# 문자 → 정수 (string to int)
stoi = {c: i for i, c in enumerate(chars)}
# 정수 → 문자 (int to string)
itos = {i: c for i, c in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return ''.join([itos[i] for i in l])

print(encode("hii there"))
print(decode(encode("hii there")))   # 원문으로 복원되어야 함

## 1.4 텐서로 변환

**텐서**는 PyTorch의 숫자 덩어리 그릇. numpy 배열과 비슷하지만 두 가지가 다르다.
1. GPU에 올려 병렬 계산이 가능
2. 미분을 자동으로 추적 (autograd)

`dtype=torch.long` : 64비트 정수. 우리 데이터는 0~64 정수이므로 정수형.
(정수 자리에 float을 넣거나 반대로 하는 것이 PyTorch 초보 에러의 큰 비중을 차지한다.)

`.shape` : 텐서의 모양. **디버깅의 90%는 shape 추적이다.** 습관을 들일 것.

In [ ]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)

print(data.shape, data.dtype)
print(data[:20])
print(decode(data[:20].tolist()))   # 원문 앞부분과 같은지 확인

## 1.5 train / validation 분리

전체를 학습에 쓰면 모델이 텍스트를 **통째로 외워버릴 수** 있다 (과적합).
일부를 떼어 학습에 쓰지 않고, "처음 보는 데이터에도 잘하는가"를 확인하는 용도로 쓴다.

In [ ]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(len(train_data), len(val_data))

## 1.6 x와 y — 학습 예제 만들기

`block_size` = 모델이 참고할 수 있는 **최대 문맥 길이** (context length).
진짜 GPT의 "컨텍스트 윈도우"의 축소판.

**핵심**: `x`와 `y`는 "입력 하나, 정답 하나"가 **아니다.**
→ **입력 8개 묶음**과 **정답 8개 묶음**이다.

```
x = data[0:8]   # 인덱스 0~7
y = data[1:9]   # 인덱스 1~8  (한 칸 밀림)

t=0: 입력 x[:1] = [18]        → 정답 y[0] = 47
t=1: 입력 x[:2] = [18,47]     → 정답 y[1] = 56
...
```

8개 입력이 앞부분을 공유하므로 제일 긴 것 하나만 저장하고 잘라 쓴다.

**`y`를 한 칸 미는 것만으로 "다음 문자 예측" 문제가 자동 생성된다.**
사람이 라벨을 붙일 필요가 없다 → **self-supervised learning (자기지도학습)**.
텍스트 자체가 정답지를 품고 있다는 것이, 인터넷 텍스트만으로 GPT를 학습시킬 수 있는 이유.

문맥 1개짜리부터 8개짜리까지 전부 동시에 학습하는 이유:
**생성할 때 처음에는 문맥이 1글자뿐**이므로, 짧은 문맥에서도 예측할 줄 알아야 한다.

In [ ]:
block_size = 8

x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"입력 {context.tolist()} → 정답 {target}")

## 1.7 배치 만들기

GPU는 **병렬 계산 기계**다. 조각 하나씩 넣으면 GPU가 논다.
여러 조각을 동시에 밀어넣는 묶음이 **배치**.

배치 안의 조각들은 **서로 아무 관계가 없다.** 텍스트 여기저기서 무작위로 뜬 것이고,
서로 소통하지 않는다. 오직 GPU 효율을 위해 같이 태운 것.

**shape 표기 (앞으로 계속 사용)**
```
(B, T, C)
 │  │  └── Channel : 각 위치를 표현하는 벡터 길이 (n_embd)
 │  └───── Time    : 문맥 길이 (block_size)
 └──────── Batch   : 동시에 처리하는 조각 수
```

- `len(data) - block_size` : 끝에서 block_size개 안쪽에서 시작하면 조각을 못 채운다
- `(batch_size,)` 의 쉼표 : 원소 1개짜리 **튜플** 문법. PyTorch는 shape을 튜플로 받는다
- `torch.stack` : 텐서 여러 개를 쌓아 **새 차원**을 만든다 (`cat`은 기존 차원을 늘림)

In [ ]:
torch.manual_seed(1337)   # 난수 고정 → 재현성 확보
batch_size = 4

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))   # 시작 위치 B개
    x = torch.stack([data[i:i+block_size]     for i in ix])     # (B, T)
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])     # (B, T) 한 칸 밀림
    return x, y

xb, yb = get_batch('train')
print(xb.shape)   # (B, T) = (4, 8)
print(xb)
print(decode(xb[0].tolist()))

---
# 2. Bigram 모델 — 기준선

**Bigram = 직전 글자 하나만 보고 다음 글자를 예측.**
앞의 7글자는 완전히 무시한다.

일부러 이 한계에서 시작한다. **문맥을 못 쓴다는 것이 이 모델의 한계이고,
attention이 정확히 그것을 해결하는 장치**이기 때문이다.

## 2.1 nn.Module — 모델을 담는 상자

- `__init__` : 어떤 부품을 쓸지 정의
- `forward` : 데이터가 어떻게 흘러갈지 정의

## 2.2 텐서 두 종류를 구분할 것

| 텐서 | shape | 내용 | decode 가능? |
|---|---|---|---|
| `xb`, `idx`, `data` | (B,T) | **문자 번호** (0~64 정수) | ✅ |
| `logits` | (B,T,C) | 후보 65개의 **점수** (실수) | ❌ |
| `probs` | (B,T,C) | 후보 65개의 **확률** | ❌ |

`logits`/`probs`에서는 **값이 아니라 몇 번째 자리냐가** 문자를 가리킨다.

In [ ]:
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # (65, 65) 표. 정수를 넣으면 그 번호의 행을 통째로 꺼내준다.
        # Bigram에서는 이 65개 숫자가 곧바로 "다음 문자 후보 65개의 점수"가 된다.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)   # (B,T) → (B,T,C=65)

        if targets is None:
            loss = None
        else:
            # cross_entropy는 (B,T,C) 3차원을 받지 않는다. (N,C) 2차원으로 펴야 한다.
            # .view()는 데이터는 그대로 두고 모양만 재해석한다.
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)   # 내부에서 softmax가 적용됨

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # autoregressive 생성: 한 글자 뱉고 → 입력 끝에 붙이고 → 또 뱉고
        for _ in range(max_new_tokens):
            logits, loss = self(idx)                            # (B,T,C)
            logits = logits[:, -1, :]                           # 마지막 시점만 (B,C)
            probs = F.softmax(logits, dim=-1)                   # 점수 → 확률 (B,C)
            idx_next = torch.multinomial(probs, num_samples=1)  # 확률대로 1개 뽑기 (B,1)
            idx = torch.cat((idx, idx_next), dim=1)             # 뒤에 이어붙임 (B,T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

### 초기 loss 검산 — 실무 디버깅 기법

아무것도 모르는 모델은 65개 후보에 확률을 균등하게 1/65씩 준다.
cross_entropy는 정답 확률에 `-log`를 씌운 값이므로:

$$-\ln(1/65) = \ln(65) \approx 4.17$$

**학습 시작 시 loss가 이 근처가 아니면 무언가 잘못된 것이다.**
(데이터가 꼬였거나 shape이 틀렸거나)

실제 측정값이 이보다 약간 높은 것은 임베딩 표가 랜덤 초기화되어
확률이 균등하지 않고 제멋대로 쏠려 있기 때문이다.

### grad_fn — autograd의 실체

출력에 `grad_fn=<NllLossBackward0>` 같은 꼬리표가 붙는다.
**"이 값이 어떤 연산으로 만들어졌는지"** 를 기록한 것.

```
임베딩 표 → logits → view → cross_entropy → loss
                                              ↑
                              loss.backward() 를 호출하면
                              ← ← ← 꼬리표를 역순으로 타고 올라가며
                                    각 파라미터의 gradient를 계산
```

이 자동 미분이 PyTorch를 쓰는 이유의 절반이다.

In [ ]:
import math
print("이론값 ln(65) =", math.log(65))
print("실제 loss     =", loss.item())

print(loss.grad_fn)
print(loss.grad_fn.next_functions)   # 한 단계 앞의 연산

## 2.3 학습 전 생성 — 기준점 확보

학습 후 결과와 비교할 기준이므로 반드시 봐 둔다.
65개 문자가 골고루 나오고 문법 구조가 전혀 없는 것이 정상이다.

`torch.zeros((1,1))` 로 시작하는 이유: 0번 문자가 줄바꿈이므로
"새 줄에서 시작한다"는 자연스러운 출발점.

In [ ]:
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))

## 2.4 학습 루프

```
데이터 → forward → loss → backward → 파라미터 업데이트 → 반복
```

**순서 고정: `zero_grad` → `backward` → `step`**

PyTorch는 gradient를 **덮어쓰지 않고 누적**한다.
`zero_grad`를 빼먹으면 이전 배치의 gradient와 섞인다. (초보 실수 1위)

- `loss.backward()` : 어느 방향으로 가야 하는지 계산
- `optimizer.step()` : 그 방향으로 실제로 한 걸음
- `lr` (learning rate) : 걸음 크기. 너무 크면 널뛰고 너무 작으면 안 움직인다

In [ ]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')

    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)   # 청소
    loss.backward()                          # 미분
    optimizer.step()                         # 전진

print(loss.item())

In [ ]:
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=300)[0].tolist()))

### 관찰

띄어쓰기 리듬, `th`/`ou`/`er` 등 흔한 문자 조합, 대문자+콜론+줄바꿈의 화자 형식이 나타난다.
그러나 **진짜 영단어는 거의 없다.**

**왜 여기가 천장인가**: Bigram은 직전 글자 하나만 본다.

```
"To be or not to b" → ?
```

모델이 보는 것은 오직 `'b'` 하나. 앞의 16글자는 존재하지도 않는다.
아무리 오래 학습해도 크게 나아지지 않는다. 파라미터가 부족한 것이 아니라
**정보가 들어오지 않는 것**이기 때문이다.

1.6절에서 애써 준비한 문맥 8개를 전부 버리고 있다.

---
# 3. attention에 도달하기

**풀어야 할 문제**

> 각 위치에서 자기 **앞에 있는 모든 위치**의 정보를 끌어오되,
> ① 미래는 못 보게 하고 (정답 커닝 방지)
> ② 중요도를 다르게 주려면?

바로 attention을 도입하지 않고, **가장 단순한 버전부터 4단계로 진화**시킨다.
각 버전이 앞 버전의 결함을 하나씩 해결한다.

실험용 장난감 텐서로 원리만 확인한다.

In [ ]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2      # C=2로 잡은 것은 눈으로 보기 편하려고
x = torch.randn(B, T, C)
print(x.shape)

## 버전 1 — for문 평균

각 위치에서 자기 자신과 그 앞의 모든 위치를 평균낸다. 뒤는 절대 보지 않는다.

```
위치 0: x[0] 의 평균          = x[0]
위치 1: x[0], x[1] 의 평균
위치 2: x[0], x[1], x[2] 의 평균
```

In [ ]:
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]              # (t+1, C) — 0번부터 t번까지
        xbow[b, t] = torch.mean(xprev, 0)

print(x[0])
print(xbow[0])   # 첫 줄은 x[0]과 동일 (자기 자신만 평균)

**진전**: Bigram은 직전 하나만 봤지만 이제 앞의 전부를 본다.

**한계**: 이중 for문 → 완전히 순차 처리 → **GPU가 놀고 있다.**

실제 학습에서는 `B=64, T=256` 이므로 배치 하나당 16,384번.
파이썬 for문으로는 불가능하다.

> **딥러닝 코드의 철칙: for문을 행렬 연산으로 바꿔라 (벡터화)**

## 버전 2 — 행렬 곱으로 벡터화

먼저 작은 예제로 원리를 본다.

- `torch.tril` : lower triangular (하삼각). 대각선 위쪽을 0으로.
  **이 삼각형 모양 하나가 "미래 차단" 장치다.**
- 행별 합을 1로 나누면 → **가중치**가 된다
- `@` : 행렬 곱

In [ ]:
a = torch.tril(torch.ones(3, 3))
a = a / a.sum(1, keepdim=True)
b_toy = torch.randint(0, 10, (3, 2)).float()
c = a @ b_toy

print('a (가중치) =\n', a)
print('b (내용물) =\n', b_toy)
print('c (결과)   =\n', c)
# c[0] = b[0]
# c[1] = (b[0]+b[1]) / 2
# c[2] = (b[0]+b[1]+b[2]) / 3

### `a`의 한 행 `[0.5, 0.5, 0]` 이 의미하는 것

```
c[1] = 0.5 × b[0]  +  0.5 × b[1]  +  0 × b[2]
        ↑              ↑              ↑
     0번을 50%      1번을 50%      2번(미래)은 0%
```

**세 숫자는 "b의 몇 번째를 얼마나 쓸 것인가"** 즉 가중치다.

| | 0번 참조 | 1번 참조 | 2번 참조 | 의미 |
|---|---|---|---|---|
| `c[0]` | 1.0 | 0 | 0 | 자기만 봄 |
| `c[1]` | 0.5 | 0.5 | 0 | 앞 2개를 반반 |
| `c[2]` | 0.33 | 0.33 | 0.33 | 앞 3개를 균등하게 |

**지금은 값이 전부 균등하다.** 이 숫자들을 균등하지 않게 만들면 어떻게 될까?

```
c[2] = 0.1 × b[0]  +  0.7 × b[1]  +  0.2 × b[2]
```

"1번 위치가 제일 중요하다"고 말하는 것이 된다. → **이것이 attention이다.**

In [ ]:
# 이제 실제 데이터에 적용. for문 결과와 같은지 검증한다.
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x                        # (T,T) @ (B,T,C) → (B,T,C)  broadcasting

print("최대 차이:", (xbow - xbow2).abs().max().item())
print(torch.allclose(xbow, xbow2, atol=1e-6))

> **부동소수점 오차 주의**
> 기본 허용오차(`1e-8`)로는 `False`가 나올 수 있다. 계산이 틀린 것이 아니다.
> for문은 `(a+b+c)÷3`, 행렬 곱은 `0.333a+0.333b+0.333c` 로 **계산 순서가 다르고**,
> 부동소수점에서는 덧셈의 결합법칙이 성립하지 않는다.
>
> 딥러닝에서는 "정확히 같은가"가 아니라 **"충분히 가까운가"** 를 묻는다.
> 값이 이상하면 **차이의 크기를 먼저 찍어보는 습관**을 들일 것.

**버전 2의 한계**: 1로 채우고 나누는 구조라 **균등 평균밖에 만들 수 없다.**

## 버전 3 — softmax 형태로 전환

**결과는 버전 2와 완전히 동일하다.** 형태만 바꾼다.

- `masked_fill(tril == 0, -inf)` : 미래 자리를 `-무한대`로
- `softmax` : `exp(점수) / exp들의 합`. 핵심은 **`exp(-inf) = 0`**
  → 미래 자리는 자동으로 0이 되고, 남은 것끼리 합이 1로 정규화된다

In [ ]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))                                  # 모든 위치에 동일한 "점수"
wei = wei.masked_fill(tril == 0, float('-inf'))            # 미래 차단
wei = F.softmax(wei, dim=-1)                               # 점수 → 확률
xbow3 = wei @ x

print(wei)
print(torch.allclose(xbow, xbow3, atol=1e-6))

### 왜 굳이 이렇게 바꾸는가 — 여기가 핵심

버전 2 (`wei / wei.sum()`) 는 구조적으로 균등 평균만 만들 수 있다.

버전 3에서는 `wei`가 **"점수"로 시작**한다.
지금은 0으로 채워서 우연히 균등할 뿐, **저 자리에 다른 숫자를 넣으면 가중치가 달라진다.**

```python
wei = torch.zeros((T,T))      # 균등
wei = 어떤_계산_결과            # ← 여기를 바꾸면?
```

예를 들어 한 행이 `[2.0, 5.0, -inf]` 라면 softmax 후 `[0.047, 0.953, 0]`.
**1번 위치를 95% 주목하는 상태**가 된다.

즉 버전 3은 **"가중치를 자유롭게 만들 수 있는 형태"로 갈아탄 것**이다.
그리고 `-inf` 마스킹은 그대로 남아 **어떤 점수가 들어와도 미래는 여전히 차단된다.**

남은 질문은 하나: **저 점수를 무엇으로 계산할 것인가?**

## 버전 4 — self-attention

**점수 = query와 key의 내적 (dot product)**

두 벡터가 얼마나 잘 맞는지를 숫자 하나로 뽑아내는 연산.
방향이 비슷하면 크게, 반대면 음수, 직교하면 0.

### Q, K, V

각 위치가 세 가지를 내놓는다.

| | 역할 |
|---|---|
| **Key** | "나는 이런 것을 갖고 있다" — 자기를 광고하는 명찰 |
| **Query** | "나는 이런 것을 찾는다" — 요구사항 |
| **Value** | "내가 실제로 전달할 내용" |

**query와 key의 내적이 크면 = 서로 잘 맞는다 = 주목한다.**

세 가지 모두 `nn.Linear`로 **학습된다.** 무엇을 광고하고 무엇을 찾을지를
데이터로부터 스스로 익힌다.

"self"인 이유: Q, K, V가 전부 **같은 x**에서 나온다.

### transpose가 필요한 이유

```
q: (B, T, 16)
k: (B, T, 16) → transpose(-2,-1) → (B, 16, T)
곱하면                            → (B, T, T)
```

결과 `(T,T)`의 `[i][j]` 자리 = **위치 i의 query와 위치 j의 key의 내적**.
모든 쌍의 내적이 한 번에 계산된다.

In [ ]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

head_size = 16
key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)      # (B, T, 16)
q = query(x)    # (B, T, 16)

wei = q @ k.transpose(-2, -1) * head_size**-0.5   # (B,T,T)  ← 내적 + scaling

tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))   # 버전 3과 동일
wei = F.softmax(wei, dim=-1)                      # 버전 3과 동일

v = value(x)    # (B, T, 16)
out = wei @ v   # (B, T, 16)

print(out.shape)
print(wei[0])

### wei 출력에서 읽을 것

1. **하삼각 유지** — 오른쪽 위가 전부 0. 미래 차단 작동 중
2. **각 행의 합 = 1** — 여전히 가중평균
3. **값이 균등하지 않다** ← 이것이 핵심

버전 3이라면 4행이 `[0.2, 0.2, 0.2, 0.2, 0.2]`여야 하지만,
실제로는 특정 위치에 크게 쏠리거나 여러 곳에 불균등하게 분산된다.

**각 위치가 앞의 어디를 얼마나 볼지 스스로 정한 결과다.**

4. `grad_fn`이 붙어 있다 → **이 가중치들이 학습 대상**이라는 뜻.
   학습이 진행되면 "모음은 앞의 자음을 주목한다" 같은 규칙이 저절로 생긴다.

### scaling — 왜 √head_size로 나누는가

**softmax는 입력값이 크면 극단적으로 한 곳에 쏠린다** (`exp` 때문).

In [ ]:
a = torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])
print(F.softmax(a,     dim=-1))   # 완만
print(F.softmax(a * 8, dim=-1))   # 한 곳에 쏠림

In [ ]:
# q·k 내적은 차원 수에 비례해 분산이 커진다
k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)

print("scaling 없음:", (q @ k.transpose(-2,-1)).var().item())                    # ≈ head_size
print("scaling 있음:", (q @ k.transpose(-2,-1) * head_size**-0.5).var().item())  # ≈ 1

분산이 16이면 값이 ±4~5로 널뛰고, 그 상태로 softmax를 태우면
**학습 시작부터 한 위치만 보는 극단적 attention**이 된다.

`√head_size`로 나누면 분산이 1로 돌아와 완만하게 시작하고,
**학습하면서 필요한 만큼만 뾰족해지도록** 둔다.

→ 논문 제목이 *Scaled* Dot-Product Attention 인 이유.

---
# 4. 최종 Transformer

## 남은 부품 3개

### ① Multi-head
Head 하나는 **한 종류의 관계**밖에 못 본다.
Q/K/V 가중치가 Head마다 독립적으로 학습되므로, 여러 개를 병렬로 두면 각자 다른 관점을 익힌다.

```
Head 1: "내 앞에 자음이 있나?"
Head 2: "이 단어가 어디서 시작했지?"
Head 3: "따옴표가 열려 있나?"
```

**`head_size = n_embd // n_head`** — n_embd를 먼저 정하고 head 수만큼 나눠 갖는다.
합치면 원래 크기로 복원되므로 총 계산량은 비슷하고 **관점만 쪼갠 것**.

> `nn.ModuleList` 필수. 그냥 파이썬 리스트에 모듈을 담으면
> PyTorch가 파라미터로 인식하지 못해 학습이 되지 않는다.

### ② FeedForward
attention은 **정보를 모으는(communication)** 연산.
모으자마자 답을 내라고 하면 **곱씹을 틈이 없다.**
FFN은 각 위치가 혼자서 그 정보를 가공하는(computation) 층.

```
attention : 위치들끼리 소통
FFN       : 각자 생각 정리
```
이 둘이 번갈아 나오는 것이 Transformer의 기본 리듬.

### ③ 잔차연결 + LayerNorm
`x + f(x)` — 덧셈의 미분은 1이므로 **gradient가 손실 없이 뒤로 흐르는 고속도로**가 생긴다.
이것이 없으면 층을 깊게 쌓을 때 gradient가 앞쪽 층에 도달하지 못한다 (gradient vanishing).

관점을 바꾸면, 각 층이 "새 표현을 만드는" 것이 아니라 **"기존 표현에 조금씩 보태는"** 역할이 된다.

`self.sa(self.ln1(x))` — 정규화를 **연산 전에** 하는 **pre-norm**.
원논문은 post-norm이었으나 pre-norm이 더 안정적이어서 현재 표준.

### shape 보존이 곧 확장성
Block의 입출력이 모두 `(B,T,n_embd)`로 같기 때문에 층을 몇 개든 쌓을 수 있다.
잔차연결 `x + f(x)` 역시 shape이 같아야 성립한다.

In [ ]:
# ---- 하이퍼파라미터 ----
batch_size = 64
block_size = 256      # 문맥 8 → 256
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384          # 32 → 384
n_head = 6            # head_size = 384 / 6 = 64
n_layer = 6           # Block 6층
dropout = 0.2

torch.manual_seed(1337)
print("device:", device)   # cuda 가 아니면 런타임 유형을 GPU로 변경할 것

In [ ]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size]     for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)      # 모델과 데이터는 같은 device에 있어야 한다

@torch.no_grad()                            # 미분 기록을 꺼서 메모리·속도 절약
def estimate_loss():
    out = {}
    model.eval()                            # dropout 끄기
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()          # 배치 하나의 loss는 들쭉날쭉하므로 평균
    model.train()                           # dropout 다시 켜기
    return out

In [ ]:
class Head(nn.Module):
    """ self-attention 한 세트 (Q/K/V 한 벌) """

    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # tril은 학습 대상이 아닌 고정 상수. 모델과 함께 저장·이동되도록 buffer로 등록
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)                                          # (B,T,hs)
        q = self.query(x)                                        # (B,T,hs)
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5         # (B,T,T)
        wei = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)                                        # (B,T,hs)
        return wei @ v                                           # (B,T,hs)


class MultiHeadAttention(nn.Module):
    """ Head 여러 개를 병렬로 """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)   # 합친 결과를 한 번 더 섞음
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)     # 채널 방향으로 이어붙임
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    """ 위치별 가공. 넓혔다가(×4) 다시 좁힌다 """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """ Transformer 블록: 소통(attention) → 가공(FFN) """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa   = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))      # 잔차연결 + pre-norm
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        # position embedding이 필요한 이유:
        # wei @ v 는 가중합이라 순서 정보가 없다. 없으면 "abc"와 "cba"를 구분 못 한다.
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks  = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f    = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)   # 최종적으로 65개 점수로 변환

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)                        # 무슨 글자인가
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))  # 몇 번째인가
        x = tok_emb + pos_emb                                            # (B,T,n_embd)
        x = self.blocks(x)                                               # (B,T,n_embd)
        x = self.ln_f(x)
        logits = self.lm_head(x)                                         # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits  = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # position embedding 테이블이 block_size까지만 있으므로 잘라야 한다.
            # 이것이 곧 컨텍스트 윈도우의 물리적 한계다.
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


model = GPTLanguageModel().to(device)
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

## 학습

Colab T4 GPU 기준 약 20~30분.

사양이 부족하면 `n_layer=4, n_embd=192, block_size=128` 로 축소하면
5분 내외로 끝나며 val loss 1.7 수준을 얻을 수 있다.

**step 0의 loss가 4.17(=ln 65) 근처인지 확인할 것.**

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=1000)[0].tolist()))

---
# 5. 결과 해석

## loss 추이

| 단계 | val loss |
|---|---|
| 학습 전 (랜덤) | 5.03 |
| Bigram | 2.38 |
| self-attention 1 head | 2.29 |
| **Transformer (6층 6 head)** | **1.47** |

## 과적합 관측

```
step 4000: train 1.1222, val 1.4786
step 4500: train 1.0845, val 1.4696   ← val 최저
step 4999: train 1.0503, val 1.4898   ← val 상승
```

train은 계속 하락하는데 **val은 4,500 스텝 부근에서 최저를 찍고 반등**한다.
두 값의 격차도 벌어진다 (1.05 vs 1.49).

**과적합이 시작된 신호.** 10.8M 파라미터로 약 100만 자를 학습하므로
모델이 일반화하는 대신 암기하기 시작한 것.

개선 여지:
- val 최저 시점의 모델을 저장 → **early stopping**
- dropout 상향, 데이터 증가, 모델 축소

## 생성 결과에서 확인되는 것

- 희곡 구조 (화자 대문자 + 콜론 + 줄바꿈)
- 셰익스피어 **실존 인물명** (JULIET, POLIXENES, HASTINGS, Rosaline, Claudio)
- 고어체 어휘 (`thou`, `thee`, `Beseech`)
- 아포스트로피 축약형의 정확한 위치
- 한 문장 안에서 유지되는 문맥 → `block_size=256`이 작동한 결과

**한계**: 의미가 없다. 존재하지 않는 단어가 섞이고 문장 간 논리가 없다.
문자 단위 + 10.8M 파라미터 + 100만 자로는 여기가 한계.

진짜 GPT는 토큰 단위에 파라미터 수천억 개, 데이터는 수조 토큰이지만
**구조는 위와 동일하다.** 규모만 다를 뿐이다.

---
# 6. 자가진단 — 2회차 재구현 전 체크리스트

아래를 **코드를 보지 않고** 할 수 있는지 확인한다.

- [ ] `get_batch` 함수를 백지에서 쓸 수 있다
- [ ] `(B,T,C)`가 각각 무엇인지 즉시 말할 수 있다
- [ ] `tril` + `-inf` + `softmax`가 왜 그 순서인지 설명할 수 있다
- [ ] Q/K/V의 역할을 비유 없이 설명할 수 있다
- [ ] `q @ k.transpose(-2,-1)`에서 transpose가 왜 필요한지 shape으로 설명할 수 있다
- [ ] scaling(`* head_size**-0.5`)이 없으면 무슨 일이 생기는지 말할 수 있다
- [ ] position embedding이 없으면 무슨 일이 생기는지 말할 수 있다
- [ ] `zero_grad`를 빼먹으면 무슨 일이 생기는지 말할 수 있다
- [ ] 버전 2가 아니라 버전 3의 형태여야 하는 이유를 말할 수 있다

**막힌 항목을 여기에 기록:**

```
(2회차에서 집중할 지점)
```

---

# 7. 다음 단계

- [ ] 코드를 보지 않고 백지에서 재구현 (2회차)
- [ ] 문자 단위 → BPE 토크나이저로 교체
- [ ] 한국어 데이터셋으로 학습
- [ ] early stopping / 체크포인트 저장 추가
- [ ] attention 가중치 시각화

## 참고
- Vaswani et al., *Attention Is All You Need* (2017)
- Andrej Karpathy, nanoGPT
- 데이터: Tiny Shakespeare (약 1.1MB, 65개 문자)